# Revisión multiclass y falsos negativos — OOF del 05 (protocolo C)

Sin reentrenar: revisamos el mejor modelo OOF del 05 por `Multiclass.Label` solo en hojas `unhealthy` (sensibilidad y FN), con énfasis en `rust_level_1`. Prefijo de resultados: `rocole_d06_`.


Librerías del cuaderno en una sola celda: tablas y, si hay figura, matplotlib.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


Semilla fija del proyecto por si hace falta reproducir algo auxiliar (p. ej. orden de figura).


In [2]:
# Semilla fija del proyecto
seed = 42

seed


Carpeta de resultados D-cv-errores y rutas a los tres CSV de D-05 (métricas CV, OOF del ganador y OOF de todos los modelos).


In [3]:
results_dir = Path("../../../experiments/results/rocole/D-cv-errores")
cv_metrics_path = results_dir / "rocole_d05_cv_metrics.csv"
best_oof_path = results_dir / "rocole_d05_best_oof_predictions.csv"
all_oof_path = results_dir / "rocole_d05_oof_predictions.csv"

print("results_dir:", results_dir.resolve(), "existe:", results_dir.exists())
print("cv_metrics_path existe:", cv_metrics_path.exists())
print("best_oof_path existe:", best_oof_path.exists())
print("all_oof_path existe:", all_oof_path.exists())
results_dir, cv_metrics_path, best_oof_path, all_oof_path


results_dir: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores existe: True
cv_metrics_path existe: True
best_oof_path existe: True
all_oof_path existe: True


Tabla de métricas CV de D-05 para saber quién ganó por `f1_macro` (sin reabrir el holdout histórico).


In [4]:
cv_metrics = pd.read_csv(cv_metrics_path)

cv_metrics


Nombre del ganador según la media CV ya guardada en D-05.


In [5]:
best_model_name = cv_metrics.loc[cv_metrics["mean_f1_macro"].idxmax(), "model_name"]

best_model_name


Predicciones OOF del ganador (File, planta, etiquetas, fold, `y_true`, `y_pred`, `model_name`).


In [6]:
best_oof_predictions = pd.read_csv(best_oof_path)

print("filas OOF ganador:", len(best_oof_predictions))
best_oof_predictions.head()


filas OOF ganador: 1481


Comprobamos que `model_name` en el CSV coincide con el ganador de métricas y cuántas hojas hay en el OOF.


In [7]:
oof_model_names = best_oof_predictions["model_name"].unique()
print("model_name en OOF:", oof_model_names)
print("best_model_name:", best_model_name)
print("coincide:", set(oof_model_names) == {best_model_name})
print("n_hojas OOF:", len(best_oof_predictions))


model_name en OOF: <StringArray>
['random_forest']
Length: 1, dtype: str
best_model_name: random_forest
coincide: True
n_hojas OOF: 1481


Solo hojas enfermas: el recall por enfermedad se mide sobre `Binary.Label` / `y_true` == `unhealthy`.


In [8]:
unhealthy_oof_leaves = best_oof_predictions.loc[
    best_oof_predictions["y_true"] == "unhealthy"
].copy()

print("n_unhealthy:", len(unhealthy_oof_leaves))
unhealthy_oof_leaves.head()


n_unhealthy: 723


Inventario de enfermas por `Multiclass.Label` (red_spider_mite, rust_level_1..4); contrastamos `rust_level_1` con las 333 del índice completo.


In [9]:
unhealthy_multiclass_counts = (
    unhealthy_oof_leaves["Multiclass.Label"]
    .value_counts()
    .rename_axis("Multiclass.Label")
    .reset_index(name="n_leaves")
)

unhealthy_multiclass_counts


Por cada `Multiclass.Label`: total, TP (`y_pred` == unhealthy), FN (`y_pred` == healthy).


In [10]:
multiclass_detection_counts = (
    unhealthy_oof_leaves.groupby("Multiclass.Label", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        tp=("y_pred", lambda s: (s == "unhealthy").sum()),
        fn=("y_pred", lambda s: (s == "healthy").sum()),
    )
)

multiclass_detection_counts


Sensibilidad = TP / (TP + FN) por `Multiclass.Label`.


In [11]:
multiclass_sensitivity = multiclass_detection_counts.copy()
multiclass_sensitivity["sensitivity"] = (
    multiclass_sensitivity["tp"]
    / (multiclass_sensitivity["tp"] + multiclass_sensitivity["fn"])
)

multiclass_sensitivity


Fila de `rust_level_1`: n, TP, FN y sensibilidad; es el foco del cuaderno (leve, fácil de confundir con sana).


In [12]:
rust_level_1_sensitivity = multiclass_sensitivity.loc[
    multiclass_sensitivity["Multiclass.Label"] == "rust_level_1"
].copy()

rust_level_1_sensitivity


Contexto opcional: misma sensibilidad multiclass con las columnas OOF de Random Forest (no elige ganador).


In [13]:
all_oof_predictions = pd.read_csv(all_oof_path)

print("filas all OOF:", len(all_oof_predictions))
all_oof_predictions.head()


filas all OOF: 1481


Filtramos enfermas en el CSV de todos los modelos (misma lógica que el ganador).


In [14]:
unhealthy_all_oof_leaves = all_oof_predictions.loc[
    all_oof_predictions["Binary.Label"] == "unhealthy"
].copy()

print("n_unhealthy all OOF:", len(unhealthy_all_oof_leaves))
unhealthy_all_oof_leaves.head()


n_unhealthy all OOF: 723


Por cada `Multiclass.Label` en el OOF de RF: total, TP (`pred_random_forest` == unhealthy), FN (`pred_random_forest` == healthy).


In [15]:
forest_multiclass_detection_counts = (
    unhealthy_all_oof_leaves.groupby("Multiclass.Label", as_index=False)
    .agg(
        n_leaves=("File", "count"),
        tp=("pred_random_forest", lambda s: (s == "unhealthy").sum()),
        fn=("pred_random_forest", lambda s: (s == "healthy").sum()),
    )
)

forest_multiclass_detection_counts


Sensibilidad = TP / (TP + FN) por `Multiclass.Label` con las predicciones RF OOF (solo contraste).


In [16]:
forest_multiclass_sensitivity = forest_multiclass_detection_counts.copy()
forest_multiclass_sensitivity["sensitivity"] = (
    forest_multiclass_sensitivity["tp"]
    / (forest_multiclass_sensitivity["tp"] + forest_multiclass_sensitivity["fn"])
)

forest_multiclass_sensitivity


Persistimos la tabla de sensibilidad del ganador con prefijo `rocole_d06_`.


In [17]:
sensitivity_csv_path = results_dir / "rocole_d06_multiclass_sensitivity.csv"
multiclass_sensitivity.to_csv(sensitivity_csv_path, index=False)

print("guardado:", sensitivity_csv_path.resolve())
sensitivity_csv_path


guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d06_multiclass_sensitivity.csv


Persistimos la tabla de sensibilidad del RF (contraste) con prefijo `rocole_d06_`.


In [18]:
forest_sensitivity_csv_path = results_dir / "rocole_d06_forest_multiclass_sensitivity.csv"
forest_multiclass_sensitivity.to_csv(forest_sensitivity_csv_path, index=False)

print("guardado:", forest_sensitivity_csv_path.resolve())
forest_sensitivity_csv_path


guardado: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d06_forest_multiclass_sensitivity.csv


Figura: sensibilidad por `Multiclass.Label` del ganador, destacando `rust_level_1`.


In [19]:
sensitivity_figure_path = results_dir / "rocole_d06_sensitivity_by_multiclass.png"

plot_order = [
    "red_spider_mite",
    "rust_level_1",
    "rust_level_2",
    "rust_level_3",
    "rust_level_4",
]
plot_table = (
    multiclass_sensitivity.set_index("Multiclass.Label")
    .reindex(plot_order)
    .dropna(subset=["sensitivity"])
    .reset_index()
)

bar_colors = [
    "#c45c26" if label == "rust_level_1" else "#4a7c59"
    for label in plot_table["Multiclass.Label"]
]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(plot_table["Multiclass.Label"], plot_table["sensitivity"], color=bar_colors)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Sensibilidad (recall)")
ax.set_xlabel("Multiclass.Label")
ax.set_title(f"Sensibilidad OOF por multiclass — {best_model_name}")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(sensitivity_figure_path, dpi=150)
plt.close("all")

print("figura:", sensitivity_figure_path.resolve())
sensitivity_figure_path


figura: D:\Maestria\Documentos\Proyecto de innovación tecnologica en IA\Desarrollo\diagnostico-hojas-cafe\experiments\results\rocole\D-cv-errores\rocole_d06_sensitivity_by_multiclass.png


## Conclusiones

Con `random_forest` (ganador del 05), el binario OOF detecta bien roya avanzada y ácaro (`rust_level_2` sensibilidad 0.84, `rust_level_3` 0.98, `rust_level_4` 1.00, `red_spider_mite` 0.91), pero `rust_level_1` queda en ~0.67 (224 TP / 109 FN sobre 333).

Falsos negativos (unhealthy→healthy) en ese OOF: 109 en roya 1, 25 en roya 2, 14 en ácaro, 1 en roya 3 y 0 en roya 4 (149 en total). Casi tres de cada cuatro FN son roya leve.

Hipótesis (sin medir lesión aparte): la roya nivel 1 deja poco cambio de color/textura frente a una hoja sana, y el modelo binario las confunde; a mayor severidad, más señal y menos FN. No abrimos un cuaderno solo de FN: el desglose multiclass ya localiza el error.
